# 3B karaciger VAE — yerel egitim (RTX 3080 Ti)

Bu notebook 3B sekil-VAE'sini **kendi GPU'nda** egitir. Sira:
1. Ortam kontrolu (dogru cekirdek + CUDA)
2. Smoke precompute (32^3, val split — hizli)
3. Smoke egitim (32^3, birkac epoch — 'calisiyor mu?')
4. Tam egitim (64^3, early-stopping)
5. Sonuc ozeti

> Hucreleri **sirayla** calistir. Yeni kod yok — hepsi `scripts/` altindaki
> mevcut `precompute_volumes.py` ve `train_vae3d.py`'yi cagirir.

## Hucre 1 — Ortam kontrolu

En kritik kontrol: **dogru cekirdek secili mi?** Sistemde iki Python var —
PATH'teki duz `python` CPU-only, CUDA yalnizca **sam2** conda ortaminda.
Asagidaki hucre `sys.executable`'i yazdirir; CUDA gorunmuyorsa buyuk bir
uyari basar -> o zaman **Kernel menusunden sam2 cekirdegini sec** ve tekrar
calistir.

In [ ]:
import os, sys
from pathlib import Path

import torch

# --- Proje kokunu bul: notebook notebooks/ altindaysa bir ust dizine cik ---
here = Path.cwd()
project_root = here.parent if here.name == "notebooks" else here
os.chdir(project_root)
sys.path.insert(0, str(project_root))
print("proje koku :", project_root)
assert (project_root / "scripts" / "train_vae3d.py").exists(), \
    "train_vae3d.py bulunamadi — notebook proje kokunden veya notebooks/ altindan acilmali"

# --- Dogru cekirdek + CUDA guvenlik kontrolu ---
print("python     :", sys.executable)   # 'sam2' gecmeli
print("torch      :", torch.__version__)
cuda_ok = torch.cuda.is_available()
print("CUDA var mi:", cuda_ok)
if cuda_ok:
    print("GPU        :", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU bellek : {total_gb:.1f} GB")
else:
    print("\n" + "!" * 68)
    print("!! UYARI: CUDA GORUNMUYOR — buyuk olasilikla YANLIS CEKIRDEK secili.")
    print("!! Yukaridaki 'python' yolu 'sam2' icermeli. Icermiyorsa:")
    print("!!   Kernel menusu -> sam2 conda ortaminin cekirdegini sec -> bu")
    print("!!   hucreyi tekrar calistir. CPU'da egitim cok yavas olur.")
    print("!" * 68)

## Hucre 2 — Smoke precompute (32^3, val split)

Hizli saglik kontrolu: sadece **val** split'ini 32^3'te NIfTI'den `.npz`'ye
cevirir (48 hasta, hizli). Script doluluk (occupancy) dagilimini ve NIfTI
parity sonucunu loglar — cok dusuk doluluk (<%1) organin resize'da eridigini
gosterirdi.

In [ ]:
import subprocess, sys

# Sadece val split, 32^3 (config varsayilani). --no-verify KOYMUYORUZ ki
# parity de gorunsun. sys.executable ile CALISAN cekirdegin python'unu
# kullaniriz (yanlis PATH python'una dusme riski yok).
cmd = [sys.executable, "scripts/precompute_volumes.py",
       "--config", "configs/vae3d.yaml", "--splits", "val"]
print(">", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

## Hucre 3 — Smoke egitim (32^3, birkac epoch)

`train_vae3d.py`'yi 32^3 + `--limit 20` ile calistirir. Amac: **loss dusuyor
mu, cokuyor mu** gormek. Bittiginde egitim egrisi ve reconstruction onizlemesini
(3 ortogonal kesit) notebook icine basar.

> `--limit` sadece hiz icin 20 hacimle sinirlar; gercek egitim Hucre 4'te.
> Cok hizli gormek istersen `configs/vae3d.yaml`'de `train.epochs`'u gecici
> 10 yapabilirsin.

In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

cmd = [sys.executable, "scripts/train_vae3d.py",
       "--config", "configs/vae3d.yaml", "--limit", "20"]
print(">", " ".join(cmd), "\n")
subprocess.run(cmd, check=True)

out = Path("outputs")
for png in sorted(out.glob("vae3d_training_curves_*.png")):
    print(png); display(Image(str(png)))
recon = sorted(out.glob("vae3d_recon_epoch_*_*.png"))
if recon:
    print(recon[-1]); display(Image(str(recon[-1])))

## Hucre 4 — Tam egitim (64^3)

**Uzun surer.** 64^3'e cikmak icin `_base_: vae3d.yaml`'i miras alip yalnizca
`input_size`'i ezen gecici bir config (`configs/vae3d_full.yaml`) yazilir —
cekirdek kod/config degismez. Sonra:
1. 64^3 precompute (tum splitler, `--limit` yok)
2. 64^3 tam egitim (epochs=100, early-stopping patience=15)

> - GPU'da uzun surebilir; **checkpoint her epoch** `outputs/checkpoints/`'a yazilir.
> - Kesilirse: asagidaki egitim satirina
>   `"--resume", "outputs/checkpoints/vae3d_last_latent64.pt"` ekleyip tekrar
>   calistir — kaldigi epoch'tan devam eder.
> - **Overfit izle:** loglardaki `gap=` (train - val total) buyur ve val recon
>   Dice dusmeye baslarsa 222 hacim az veri demektir -> augmentation gerekecek
>   (hook `train_vae3d.py`'de hazir, simdilik no-op).

In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

# --- 64^3 icin gecici tam config yaz (tek satir override, miras yoluyla) ---
full_cfg = Path("configs/vae3d_full.yaml")
full_cfg.write_text(
    "# Otomatik uretildi (yerel notebook Hucre 4): 64^3 tam egitim.\n"
    '_base_: "vae3d.yaml"\n'
    "vae3d:\n"
    "  input_size: [64, 64, 64]\n",
    encoding="utf-8",
)
print("yazildi:", full_cfg)

# --- 1) 64^3 precompute (tum splitler) ---
subprocess.run([sys.executable, "scripts/precompute_volumes.py",
                "--config", str(full_cfg)], check=True)

# --- 2) 64^3 tam egitim ---
# Kesildiyse asagiya --resume ekle:
#   + ["--resume", "outputs/checkpoints/vae3d_last_latent64.pt"]
subprocess.run([sys.executable, "scripts/train_vae3d.py",
                "--config", str(full_cfg)], check=True)

out = Path("outputs")
for png in sorted(out.glob("vae3d_training_curves_*.png")):
    print(png); display(Image(str(png)))
for png in sorted(out.glob("vae3d_recon_epoch_*_*.png"))[-2:]:
    print(png); display(Image(str(png)))

## Hucre 5 — Sonuc ozeti

Best checkpoint'ten son metrikleri okur: val recon Dice, blur, kac epoch,
checkpoint yolu. Sonraki adim **Asama 8 — 3B korelasyon** (bu VAE'nin
reconstruction hatasinin gercek Dice ile korelasyonunu olcup 2B'nin
rho=0.67'siyle karsilastirma).

In [ ]:
import torch
from pathlib import Path

ckpt_path = Path("outputs/checkpoints/vae3d_best_latent64.pt")
if not ckpt_path.exists():
    print("Henuz best checkpoint yok — once Hucre 3 (smoke) veya 4 (tam) calistir.")
else:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m = ckpt["metrics"]
    print("checkpoint     :", ckpt_path)
    print("epoch          :", ckpt["epoch"] + 1)
    print("input_size     :", ckpt["config"]["vae3d"]["input_size"])
    print("val recon Dice :", round(float(m.get("recon_dice", float('nan'))), 4))
    print("val blur       :", round(float(m.get("blur", float('nan'))), 4))
    print("best recon Dice:", round(float(m.get("best_recon_dice", float('nan'))), 4))
    print("\nSonraki adim: Asama 8 — 3B korelasyon (2B rho=0.67 ile karsilastir).")